In [0]:
%run "../00_config"

In [0]:
from pyspark.sql import functions as F
from datetime import datetime, timezone
from pyspark.sql.window import Window

##Load silver weather and gold city

In [0]:
df_silver_weather = spark.table(SILVER_WEATHER)
df_dim_city = spark.table(GOLD_CITY)
display(df_silver_weather)

##Build fact_weather

In [0]:
df_fact_weather = (df_silver_weather
   # Join to dim_city to get city_id
   .join(
       df_dim_city.select("city_id", "city"),
       on="city",
       how="left"
   )
   
    # Surrogate key 
    .withColumn("weather_id",
        F.concat(
            F.lit("WAHA1"),
            F.lpad(
                F.row_number().over(
                    Window.partitionBy(F.lit(1)).orderBy("date")
                ).cast("string"),
                5, "0"
            )
        )
    )

   .withColumn("_gold_ingested_at",
       F.to_timestamp(F.lit(datetime.now(timezone.utc).isoformat()))
   )

   .select(
       "weather_id",
       "date",
       "city_id",
       "temperature_max",
       "temperature_min",
       "apparent_temperature_max",
       "precipitation_sum",
       "windspeed_max",
       "wind_direction_dominant",
       "_gold_ingested_at"
   )
)
print(f"fact_weather rows: {df_fact_weather.count()}")
display(df_fact_weather)

##Validate join quality

In [0]:
null_city = df_fact_weather.filter(F.col("city_id").isNull()).count()
print(f"Rows with null city_id : {null_city}")

##Write to gold

In [0]:
(df_fact_weather
   .write
   .format("delta")
   .mode("overwrite")
   .saveAsTable(GOLD_WEATHER)
)
print(f"✅ {df_fact_weather.count()} rows written to {GOLD_WEATHER}")

##Validate

In [0]:
df_check = spark.table(GOLD_WEATHER)
print(f"Total rows     : {df_check.count()}")
print(f"Distinct cities: {df_check.select('city_id').distinct().count()}")
print(f"Date range     : {df_check.agg(F.min('date'), F.max('date')).collect()[0]}")
df_check.show(5, truncate=False)

In [0]:
%sql
SELECT DISTINCT city_id FROM saudi_retail_catalog.gold.fact_weather;